# Pratica 03 — Distribuzioni empiriche

**Corso:** Statistica Applicata con Python · **Lezione 03** · **Tempo stimato:** 60-90 minuti
**Notebook di lezione:** `notebooks_clase/clase03_distribuciones_empiricas_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti dice cosa rivedere. Se vedi "Completa le righe contrassegnate con TODO", manca ancora qualcosa da compilare.
- Le **soluzioni** sono in fondo, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine: alcuni esercizi riutilizzano variabili dei precedenti.
- Dati: `penguins` (*Palmer Penguins*, Horst, Hill e Gorman). Se non ci sono una copia locale né connessione, si usa un dataset sintetico (*esempio illustrativo*): i controlli funzionano allo stesso modo con entrambi.

---
## Cella 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)          # seed fisso: otteniamo tutti gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
sns.set_theme(style="whitegrid")
print("Librerie caricate correttamente.")

In [ ]:
def generar_penguins_sintetico(semilla=2026):
    """Genera un dataset SINTETICO con le stesse colonne di `penguins`.

    ESEMPIO ILLUSTRATIVO: i valori imitano l'ordine di grandezza dei dati reali,
    ma NON sono pinguini reali. Serve solo come alternativa senza connessione."""
    r = np.random.default_rng(semilla)
    # per specie: n, (media, deviazione) di ogni misura e isole possibili
    especies = {
        "Adelie":    dict(n=152, pico=(38.8, 2.7), prof=(18.3, 1.2), aleta=(190, 6.5), masa=(3700, 460), islas=["Torgersen", "Biscoe", "Dream"]),
        "Chinstrap": dict(n=68,  pico=(48.8, 3.3), prof=(18.4, 1.1), aleta=(196, 7.1), masa=(3733, 384), islas=["Dream"]),
        "Gentoo":    dict(n=124, pico=(47.5, 3.1), prof=(15.0, 1.0), aleta=(217, 6.5), masa=(5076, 504), islas=["Biscoe"]),
    }
    bloques = []
    for esp, c in especies.items():
        n = c["n"]
        sexo = r.choice(["Male", "Female"], size=n)
        efecto = np.where(sexo == "Male", 0.5, -0.5)          # i maschi sono un po' più grandi delle femmine
        def medida(par):
            return par[0] + efecto * par[1] + r.normal(0, par[1] * 0.8, size=n)
        bloques.append(pd.DataFrame({
            "species": esp,
            "island": r.choice(c["islas"], size=n),
            "bill_length_mm": np.round(medida(c["pico"]), 1),
            "bill_depth_mm": np.round(medida(c["prof"]), 1),
            "flipper_length_mm": np.round(medida(c["aleta"])),
            "body_mass_g": np.round(medida(c["masa"]) / 25) * 25,     # la massa reale è arrotondata a 25 g
            "sex": sexo,
        }))
    d = pd.concat(bloques, ignore_index=True)
    idx = r.choice(len(d), size=11, replace=False)               # righe 0 e 1 senza alcuna misura; altre 9 righe senza sesso (come nei dati reali)
    d.loc[idx[:2], ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]] = np.nan
    d.loc[idx[2:], "sex"] = np.nan
    return d


def cargar_penguins():
    """Carica penguins: copia locale -> download da seaborn -> sintetico."""
    for ruta in ("datos/penguins.csv", "../datos/penguins.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "copia locale (" + ruta + ")"
    try:
        return sns.load_dataset("penguins"), "download da seaborn"
    except Exception:
        return generar_penguins_sintetico(), "SINTETICO (esempio illustrativo, non sono pinguini reali)"


def completos(df, columnas):
    """Restituisce le righe senza valori mancanti in `columnas` e INFORMA di quante ne esclude."""
    sub = df.dropna(subset=columnas)
    cols = ", ".join(columnas)
    print(f"[{cols}] righe usate: {len(sub)} su {len(df)}  (escluse per valori mancanti: {len(df) - len(sub)})")
    return sub

## Esercizio 1 — Valori mancanti: contali, non nasconderli

Con il dataset `penguins` (già caricato in `df`):

- `faltantes`: Serie con il numero di valori mancanti di **ogni colonna**,
- `n_filas_completas`: quante righe non hanno alcun valore mancante,
- `col_mas_ausentes`: il nome della colonna con più valori mancanti,
- `aleta`: la lunghezza della pinna **senza valori mancanti** (una Serie; usa la funzione `completos`).

In [ ]:
df, origen = cargar_penguins()
print("Origine dei dati:", origen)

faltantes = None            # TODO: Serie con il numero di valori mancanti di ogni colonna
n_filas_completas = None    # TODO: quante righe NON hanno alcun valore mancante
col_mas_ausentes = None     # TODO: nome della colonna con più valori mancanti
aleta = None                # TODO: lunghezza della pinna senza valori mancanti (Serie), senza toccare le altre colonne

# Controllo automatico
assert None not in (n_filas_completas, col_mas_ausentes) and faltantes is not None and aleta is not None, "Completa le quattro righe contrassegnate con TODO"
assert int(faltantes.sum()) == int(df.isna().to_numpy().sum()), "faltantes deve contare i valori mancanti di ogni colonna"
assert n_filas_completas == int(df.notna().all(axis=1).sum()), "n_filas_completas: righe senza alcun valore mancante"
assert faltantes[col_mas_ausentes] == faltantes.max(), "col_mas_ausentes deve essere la colonna con più valori mancanti"
assert aleta.notna().all() and len(aleta) == int(df["flipper_length_mm"].notna().sum()), "aleta deve essere la colonna della pinna senza valori mancanti"
print("Colonna con più valori mancanti:", col_mas_ausentes, "|", "Righe complete:", n_filas_completas)
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Un istogramma a mano

Con i dodici dati del capitolo, calcola i conteggi con intervalli di ampiezza 5 (estremi 0, 5, 10, 15, 20, 25) e con ampiezza 10 (estremi 0, 10, 20, 30), e le **proporzioni** del primo caso.

In [ ]:
x = np.array([2, 3, 7, 8, 8, 9, 12, 13, 13, 14, 21, 22])

bordes = None        # TODO: estremi 0, 5, 10, 15, 20, 25 (suggerimento: np.arange)
conteos = None       # TODO: conteggi di ogni intervallo (suggerimento: np.histogram)
proporciones = None  # TODO: ogni conteggio diviso per il totale dei dati
conteos10 = None     # TODO: conteggi con intervalli di ampiezza 10 (estremi 0, 10, 20, 30)

# Controllo automatico
assert None not in (bordes, conteos, proporciones, conteos10), "Completa le quattro righe contrassegnate con TODO"
assert list(conteos) == [2, 4, 4, 0, 2], "i conteggi con ampiezza 5 devono essere [2, 4, 4, 0, 2]"
assert np.isclose(np.sum(proporciones), 1.0), "le proporzioni devono sommare 1"
assert list(conteos10) == [6, 4, 2], "con ampiezza 10 i conteggi devono essere [6, 4, 2]"
print("Con ampiezza 5 si vede il vuoto [15,20); con ampiezza 10 scompare.")
print("Esercizio 2 corretto ✔")

## Esercizio 3 — ECDF a mano

Completa la funzione `ecdf(valores)`, che restituisce i valori **ordinati** e la proporzione cumulata (1/n, 2/n, …, n/n). Poi usa gli stessi dodici dati per calcolare `p_13` (proporzione di dati ≤ 13) e `mediana_ecdf` (primo valore in cui la ECDF arriva a 0,5 o più). Infine, con la massa corporea di `penguins`, calcola la proporzione di pinguini con massa ≤ 4 000 g.

In [ ]:
def ecdf(valores):
    """Restituisce (x ordinati, proporzione cumulata) per ogni dato."""
    xs = None    # TODO: ordina i valori
    ys = None    # TODO: 1/n, 2/n, ..., n/n
    return xs, ys

xs, ys = ecdf(x)
assert xs is not None and ys is not None, "Completa la funzione (TODO)"

p_13 = None            # TODO: proporzione di dati di x che sono <= 13 (senza usare la ECDF)
mediana_ecdf = None    # TODO: primo valore di xs in cui la ECDF arriva a 0,5 o più
masa = completos(df, ["body_mass_g"])["body_mass_g"]
p_4000 = None          # TODO: proporzione di pinguini con massa <= 4000 g

# Controllo automatico
assert None not in (p_13, mediana_ecdf, p_4000), "Completa le tre righe contrassegnate con TODO"
assert list(xs) == sorted(x) and np.isclose(ys[-1], 1.0), "xs deve essere ordinato e ys deve finire a 1"
assert np.isclose(ys[0], 1 / 12), "ys deve iniziare a 1/n"
assert np.isclose(p_13, 0.75), "ECDF(13) deve essere 9/12 = 0,75"
assert mediana_ecdf == 9, "la ECDF arriva a 0,5 al sesto dato ordinato (9)"
assert np.isclose(p_4000, np.searchsorted(np.sort(masa), 4000, side="right") / len(masa)), "controlla la proporzione di massa <= 4000 g"
print("ECDF(13) =", p_13, "|", "Primo x con ECDF >= 0,5:", mediana_ecdf, "|", "Proporzione con massa <= 4000 g:", round(p_4000, 3))
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Riepilogo per gruppi

Completa `resumen_grupos(datos, col, grupo)`: restituisce un `DataFrame` con una riga per gruppo e le colonne `n` (dati **non mancanti**), `mediana`, `Q1`, `Q3` e `IQR` (= Q3 − Q1). Si verifica prima con un mini esempio con un valore mancante e poi con le specie di `penguins`.

In [ ]:
def resumen_grupos(datos, col, grupo):
    """Tabella per gruppo: n (senza valori mancanti), mediana, Q1, Q3 e IQR di `col`."""
    # TODO: costruisci il DataFrame raggruppando per `grupo`; usa count, median e quantile(0.25) / quantile(0.75)
    tabla = None
    # TODO: aggiungi la colonna IQR = Q3 - Q1
    return tabla

mini = pd.DataFrame({"g": ["a"] * 5 + ["b"] * 5, "v": [1, 2, 3, 4, 5, 10, 20, 30, 40, np.nan]})
r = resumen_grupos(mini, "v", "g")
assert r is not None, "Completa la funzione (TODO)"

# Controllo automatico
assert set(["n", "mediana", "Q1", "Q3", "IQR"]) <= set(r.columns), "mancano colonne: devono essere n, mediana, Q1, Q3, IQR"
assert (r.loc["a", "n"], r.loc["a", "mediana"], r.loc["a", "Q1"], r.loc["a", "Q3"]) == (5, 3, 2, 4), "con il mini esempio, il gruppo a deve avere n=5, mediana=3, Q1=2, Q3=4"
assert (r.loc["b", "n"], r.loc["b", "mediana"], r.loc["b", "Q1"], r.loc["b", "Q3"]) == (4, 25, 17.5, 32.5), "il gruppo b ha 4 dati validi (il mancante è escluso): mediana=25, Q1=17,5, Q3=32,5"

tabla_aleta = resumen_grupos(df, "flipper_length_mm", "species")
assert int(tabla_aleta["n"].sum()) == int(df["flipper_length_mm"].notna().sum()), "la somma di n deve essere il numero di dati non mancanti"
assert np.allclose(tabla_aleta["IQR"], tabla_aleta["Q3"] - tabla_aleta["Q1"]), "IQR = Q3 - Q1"
display(tabla_aleta)

especie_max_iqr = None    # TODO: specie (indice di tabla_aleta) con IQR maggiore
assert especie_max_iqr is not None, "Completa le righe contrassegnate con TODO"
assert tabla_aleta.loc[especie_max_iqr, "IQR"] == tabla_aleta["IQR"].max(), "la specie con IQR maggiore della pinna non coincide"
print("Specie con IQR maggiore di lunghezza della pinna:", especie_max_iqr)
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Scegliere il grafico in base alla domanda

Per ogni domanda, scrivi il grafico più adatto: `"istogramma"`, `"kde"`, `"boxplot"` o `"ecdf"`. In tre domande c'è una risposta nettamente migliore, in una ce ne sono due valide.

In [ ]:
# TODO: scegli un grafico per ogni domanda: 'istogramma', 'kde', 'boxplot' o 'ecdf'
respuestas = {
    "Che proporzione di pinguini pesa meno di 4000 g?": None,   # TODO
    "La lunghezza della pinna ha uno o due picchi?": None,   # TODO
    "Come si confrontano mediana e dispersione della massa tra 6 gruppi (specie × sesso)?": None,   # TODO
    "Come differiscono le forme della lunghezza del becco delle 3 specie, sovrapposte?": None,   # TODO
}

validas = {
    "Che proporzione di pinguini pesa meno di 4000 g?": {"ecdf"},
    "La lunghezza della pinna ha uno o due picchi?": {"istogramma", "kde"},
    "Come si confrontano mediana e dispersione della massa tra 6 gruppi (specie × sesso)?": {"boxplot"},
    "Come differiscono le forme della lunghezza del becco delle 3 specie, sovrapposte?": {"kde", "ecdf"},
}
assert None not in respuestas.values(), "Completa le quattro righe contrassegnate con TODO"
fallos = [p for p in validas if respuestas[p] not in validas[p]]
assert not fallos, "Controlla le risposte a queste domande: " + " | ".join(fallos)
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Confrontare tre specie senza ingannare

Scrivi la funzione `comparar_grupos(df, col)` che restituisca una **figura con due pannelli che condividono l'asse x**: a sinistra, una KDE per specie (`hue="species"`, `common_norm=False`); a destra, la ECDF per specie. Deve **escludere i valori mancanti in modo esplicito** (con `completos`), mettere un titolo con l'*n* usato ed etichettare l'asse x.

Poi usala con `bill_length_mm` e scrivi le tue conclusioni.

In [ ]:
def comparar_grupos(df, col):
    datos = None    # TODO: elimina i valori mancanti di `col` (usa completos)
    fig, ejes = None, None    # TODO: crea 1 riga e 2 colonne di assi che condividano l'asse x
    # TODO: KDE per specie (hue) SENZA normalizzare congiuntamente (common_norm=False) su ejes[0]
    # TODO: ECDF per specie (hue) su ejes[1]
    # TODO: metti un titolo alla figura con l'n usato (fig.suptitle) ed etichetta gli assi
    return fig

assert comparar_grupos(df, "bill_length_mm") is not None, "Completa la funzione comparar_grupos (TODO)"
fig = comparar_grupos(df, "bill_length_mm")
plt.show()

assert len(fig.axes) == 2, "la figura deve avere 2 assi"
assert fig.axes[0].get_shared_x_axes().joined(fig.axes[0], fig.axes[1]), "i due assi devono condividere l'asse x"
assert all(len(ax.lines) >= 3 for ax in fig.axes), "ogni asse deve disegnare 3 curve (una per specie)"
assert fig._suptitle is not None and fig._suptitle.get_text().strip() != "", "metti un titolo alla figura (fig.suptitle)"
assert all(ax.get_xlabel().strip() != "" for ax in fig.axes), "etichetta l'asse x dei due pannelli"
display(resumen_grupos(df, "bill_length_mm", "species"))
print("Sfida finale corretta ✔")

**Le tue conclusioni (3-4 frasi):** quale specie ha le lunghezze del becco più piccole? Le distribuzioni si sovrappongono? Quale grafico ti è stato più utile per rispondere e perché? Cosa non puoi concludere solo con questi grafici?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontare con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
df, origen = cargar_penguins()
print("Origine dei dati:", origen)

faltantes = df.isna().sum()
n_filas_completas = int(df.notna().all(axis=1).sum())
col_mas_ausentes = faltantes.idxmax()
aleta = completos(df, ["flipper_length_mm"])["flipper_length_mm"]

assert int(faltantes.sum()) == int(df.isna().to_numpy().sum()), "faltantes deve contare i valori mancanti di ogni colonna"
assert n_filas_completas == int(df.notna().all(axis=1).sum()), "n_filas_completas: righe senza alcun valore mancante"
assert faltantes[col_mas_ausentes] == faltantes.max(), "col_mas_ausentes deve essere la colonna con più valori mancanti"
assert aleta.notna().all() and len(aleta) == int(df["flipper_length_mm"].notna().sum()), "aleta deve essere la colonna della pinna senza valori mancanti"
print("Colonna con più valori mancanti:", col_mas_ausentes, "|", "Righe complete:", n_filas_completas)
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
x = np.array([2, 3, 7, 8, 8, 9, 12, 13, 13, 14, 21, 22])

bordes = np.arange(0, 30, 5)
conteos, _ = np.histogram(x, bins=bordes)
proporciones = conteos / len(x)
conteos10, _ = np.histogram(x, bins=np.arange(0, 40, 10))

assert list(conteos) == [2, 4, 4, 0, 2], "i conteggi con ampiezza 5 devono essere [2, 4, 4, 0, 2]"
assert np.isclose(np.sum(proporciones), 1.0), "le proporzioni devono sommare 1"
assert list(conteos10) == [6, 4, 2], "con ampiezza 10 i conteggi devono essere [6, 4, 2]"
print("Con ampiezza 5 si vede il vuoto [15,20); con ampiezza 10 scompare.")
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
def ecdf(valores):
    """Restituisce (x ordinati, proporzione cumulata) per ogni dato."""
    xs = np.sort(np.asarray(valores))
    ys = np.arange(1, len(xs) + 1) / len(xs)
    return xs, ys

xs, ys = ecdf(x)

p_13 = float((x <= 13).mean())
mediana_ecdf = xs[np.argmax(ys >= 0.5)]
masa = completos(df, ["body_mass_g"])["body_mass_g"]
p_4000 = float((masa <= 4000).mean())

assert list(xs) == sorted(x) and np.isclose(ys[-1], 1.0), "xs deve essere ordinato e ys deve finire a 1"
assert np.isclose(ys[0], 1 / 12), "ys deve iniziare a 1/n"
assert np.isclose(p_13, 0.75), "ECDF(13) deve essere 9/12 = 0,75"
assert mediana_ecdf == 9, "la ECDF arriva a 0,5 al sesto dato ordinato (9)"
assert np.isclose(p_4000, np.searchsorted(np.sort(masa), 4000, side="right") / len(masa)), "controlla la proporzione di massa <= 4000 g"
print("ECDF(13) =", p_13, "|", "Primo x con ECDF >= 0,5:", mediana_ecdf, "|", "Proporzione con massa <= 4000 g:", round(p_4000, 3))
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def resumen_grupos(datos, col, grupo):
    """Tabella per gruppo: n (senza valori mancanti), mediana, Q1, Q3 e IQR di `col`."""
    g = datos.groupby(grupo)[col]
    tabla = pd.DataFrame({
        "n": g.count(),
        "mediana": g.median(),
        "Q1": g.quantile(0.25),
        "Q3": g.quantile(0.75),
    })
    tabla["IQR"] = tabla["Q3"] - tabla["Q1"]
    return tabla

mini = pd.DataFrame({"g": ["a"] * 5 + ["b"] * 5, "v": [1, 2, 3, 4, 5, 10, 20, 30, 40, np.nan]})
r = resumen_grupos(mini, "v", "g")

assert set(["n", "mediana", "Q1", "Q3", "IQR"]) <= set(r.columns), "mancano colonne: devono essere n, mediana, Q1, Q3, IQR"
assert (r.loc["a", "n"], r.loc["a", "mediana"], r.loc["a", "Q1"], r.loc["a", "Q3"]) == (5, 3, 2, 4), "con il mini esempio, il gruppo a deve avere n=5, mediana=3, Q1=2, Q3=4"
assert (r.loc["b", "n"], r.loc["b", "mediana"], r.loc["b", "Q1"], r.loc["b", "Q3"]) == (4, 25, 17.5, 32.5), "il gruppo b ha 4 dati validi (il mancante è escluso): mediana=25, Q1=17,5, Q3=32,5"

tabla_aleta = resumen_grupos(df, "flipper_length_mm", "species")
assert int(tabla_aleta["n"].sum()) == int(df["flipper_length_mm"].notna().sum()), "la somma di n deve essere il numero di dati non mancanti"
assert np.allclose(tabla_aleta["IQR"], tabla_aleta["Q3"] - tabla_aleta["Q1"]), "IQR = Q3 - Q1"
display(tabla_aleta)

especie_max_iqr = tabla_aleta["IQR"].idxmax()
assert tabla_aleta.loc[especie_max_iqr, "IQR"] == tabla_aleta["IQR"].max(), "la specie con IQR maggiore della pinna non coincide"
print("Specie con IQR maggiore di lunghezza della pinna:", especie_max_iqr)
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
respuestas = {
    "Che proporzione di pinguini pesa meno di 4000 g?": "ecdf",
    "La lunghezza della pinna ha uno o due picchi?": "istogramma",
    "Come si confrontano mediana e dispersione della massa tra 6 gruppi (specie × sesso)?": "boxplot",
    "Come differiscono le forme della lunghezza del becco delle 3 specie, sovrapposte?": "kde",
}

validas = {
    "Che proporzione di pinguini pesa meno di 4000 g?": {"ecdf"},
    "La lunghezza della pinna ha uno o due picchi?": {"istogramma", "kde"},
    "Come si confrontano mediana e dispersione della massa tra 6 gruppi (specie × sesso)?": {"boxplot"},
    "Come differiscono le forme della lunghezza del becco delle 3 specie, sovrapposte?": {"kde", "ecdf"},
}
fallos = [p for p in validas if respuestas[p] not in validas[p]]
assert not fallos, "Controlla le risposte a queste domande: " + " | ".join(fallos)
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def comparar_grupos(df, col):
    datos = completos(df, [col, "species"])
    fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
    sns.kdeplot(data=datos, x=col, hue="species", common_norm=False, ax=ejes[0])
    sns.ecdfplot(data=datos, x=col, hue="species", ax=ejes[1])
    fig.suptitle(f"Lunghezza del becco per specie: n = {len(datos)}")
    for ax in ejes:
        ax.set_xlabel("Lunghezza del becco (mm)")
    return fig

fig = comparar_grupos(df, "bill_length_mm")
plt.show()

assert len(fig.axes) == 2, "la figura deve avere 2 assi"
assert fig.axes[0].get_shared_x_axes().joined(fig.axes[0], fig.axes[1]), "i due assi devono condividere l'asse x"
assert all(len(ax.lines) >= 3 for ax in fig.axes), "ogni asse deve disegnare 3 curve (una per specie)"
assert fig._suptitle is not None and fig._suptitle.get_text().strip() != "", "metti un titolo alla figura (fig.suptitle)"
assert all(ax.get_xlabel().strip() != "" for ax in fig.axes), "etichetta l'asse x dei due pannelli"
display(resumen_grupos(df, "bill_length_mm", "species"))
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*